In [9]:
import time
import random
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision.transforms as T
from torchvision.models.segmentation import deeplabv3_resnet101, DeepLabV3_ResNet101_Weights

# 0. Environment & Workspace Directory Setup
base_dir = Path("/Users/anupamahalder/Autonomous_Vehicle_Risk_Estimation_Analysis")
cam_front_dir = base_dir / "v1.0-mini/samples/CAM_FRONT/"
benchmark_images = sorted(list(cam_front_dir.glob("*.jpg")))[:100]

# Ground truth path
gt_output_dir = base_dir / "sam_pseudo_gt_masks"

# Create a dedicated directory for DeepLabV3 outputs and logs
deeplab_output_dir = base_dir / "deeplabv3/outputs"
deeplab_output_dir.mkdir(parents=True, exist_ok=True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Ground truth masks directory: {gt_output_dir}")

# 1. Load DeepLabV3-ResNet101 (Pretrained via Torchvision)
print("Loading DeepLabV3-ResNet101 model...")
weights = DeepLabV3_ResNet101_Weights.DEFAULT
model = deeplabv3_resnet101(weights=weights)
model = model.to(device)
model.eval()

# 2. Preprocessing Transformations
input_size = (384, 640)
transform = T.Compose([
    T.Resize(input_size),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Helper function to find matching mask paths precisely
# Debug: List the first 5 files in the gt_dir to inspect their exact naming
print("\n--- DEBUG: Inspecting contents of gt_dir ---")
sample_masks = list(gt_dir.glob("*"))
print(f"Total files found in mask directory: {len(sample_masks)}")
for sm in sample_masks[:5]:
    print(f"  - {sm.name}")
print("-------------------------------------------\n")

def find_matching_mask(img_path, gt_dir):
    # Try exact stem with .png
    mask_path = gt_dir / f"{img_path.stem}.png"
    if mask_path.exists():
        return mask_path
        
    # Try exact filename replacement
    mask_path_jpg = gt_dir / f"{img_path.stem}.jpg"
    if mask_path_jpg.exists():
        return mask_path_jpg

    # Search loosely if nothing else matches
    for p in gt_dir.glob("*.png"):
        if img_path.stem in p.stem:
            return p
            
    return None

# 3. Evaluation & Visualization Function
def evaluate_deeplabv3(model, image_paths, gt_dir, save_dir, device, num_classes=19):
    model.eval()
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    latencies = []
    confusion_matrix = np.zeros((num_classes, num_classes))
    valid_samples = []

    print(f"\n==============================================")
    print(f"EVALUATING MODEL: DeepLabV3-ResNet101")
    print(f"==============================================")

    with torch.no_grad():
        # Warmup run
        dummy = torch.randn(1, 3, input_size[0], input_size[1]).to(device)
        _ = model(dummy)

        for img_path in image_paths:
            mask_path = find_matching_mask(img_path, gt_dir)
            if mask_path is None or not mask_path.exists():
                continue
            
            image = Image.open(img_path).convert("RGB")
            orig_size = image.size 
            gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
            
            if gt_mask is None:
                continue

            input_tensor = transform(image).unsqueeze(0).to(device)

            start_time = time.time()
            output = model(input_tensor)['out']
            
            if device.type == "mps":
                torch.mps.synchronize()
            elif device.type == "cuda":
                torch.cuda.synchronize()
                
            elapsed = time.time() - start_time
            latencies.append(elapsed)

            # Upsample predictions back to original image dimensions
            upsampled = torch.nn.functional.interpolate(
                output, size=(orig_size[1], orig_size[0]), mode="bilinear", align_corners=False
            )
            pred = upsampled.argmax(dim=1).cpu().numpy().squeeze()

            if pred.shape != gt_mask.shape:
                gt_mask = np.array(
                    Image.fromarray(gt_mask.astype(np.uint8)).resize(
                        (pred.shape[1], pred.shape[0]), Image.NEAREST
                    )
                )

            mask = (gt_mask >= 0) & (gt_mask < num_classes)
            confusion_matrix += np.bincount(
                num_classes * gt_mask[mask].astype(int) + pred[mask].astype(int),
                minlength=num_classes ** 2,
            ).reshape(num_classes, num_classes)

            valid_samples.append((img_path, gt_mask, pred))

    if not latencies:
        print("[Error] No matching ground truth masks found. Please verify file naming inside:", gt_dir)
        return None

    avg_lat = np.array(latencies).mean()
    med_lat = np.median(np.array(latencies))
    avg_fps = 1.0 / avg_lat

    intersection = np.diag(confusion_matrix)
    union = confusion_matrix.sum(axis=1) + confusion_matrix.sum(axis=0) - intersection
    per_class_iou = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union != 0)
    mIoU = np.nanmean(per_class_iou)
    pixel_accuracy = np.sum(intersection) / np.sum(confusion_matrix) if np.sum(confusion_matrix) > 0 else 0.0

    print(f"\n--- Quantitative Results: DeepLabV3-ResNet101 ---")
    print(f"* Valid Samples Evaluated : {len(valid_samples)}")
    print(f"* Model Parameters        : {total_params:.2f} M")
    print(f"* Avg Latency             : {avg_lat * 1000:.2f} ms")
    print(f"* Median Latency          : {med_lat * 1000:.2f} ms")
    print(f"* Average FPS             : {avg_fps:.2f}")
    print(f"* Pixel Accuracy          : {pixel_accuracy * 100:.2f}%")
    print(f"* Mean IoU (mIoU)         : {mIoU * 100:.2f}%")
    print(f"----------------------------------------------")

    if valid_samples:
        viz_samples = random.sample(valid_samples, min(3, len(valid_samples)))
        fig, axes = plt.subplots(len(viz_samples), 3, figsize=(15, 4 * len(viz_samples)))
        
        if len(viz_samples) == 1:
            axes = np.expand_dims(axes, 0)

        for idx, (img_p, gt, pred) in enumerate(viz_samples):
            orig_img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)

            axes[idx, 0].imshow(orig_img)
            axes[idx, 0].set_title(f"Original: {img_p.name[:12]}...")
            axes[idx, 0].axis("off")

            axes[idx, 1].imshow(gt, cmap="nipy_spectral")
            axes[idx, 1].set_title("SAM Pseudo-GT Mask")
            axes[idx, 1].axis("off")

            axes[idx, 2].imshow(pred, cmap="nipy_spectral")
            axes[idx, 2].set_title("DeepLabV3 Prediction")
            axes[idx, 2].axis("off")

        plt.tight_layout()
        plt.savefig(save_dir / "deeplabv3_qualitative_results.png", dpi=300)
        plt.show()

    return {
        "Model": "DeepLabV3-ResNet101",
        "Params (M)": round(total_params, 2),
        "Avg Latency (ms)": round(avg_lat * 1000, 2),
        "FPS": round(avg_fps, 2),
        "Pixel Acc (%)": round(pixel_accuracy * 100, 2),
        "mIoU (%)": round(mIoU * 100, 2)
    }

# 4. Run Execution Pipeline
deeplab_metrics = evaluate_deeplabv3(model, benchmark_images, gt_output_dir, deeplab_output_dir, device)

Using device: mps
Ground truth masks directory: /Users/anupamahalder/Autonomous_Vehicle_Risk_Estimation_Analysis/sam_pseudo_gt_masks
Loading DeepLabV3-ResNet101 model...

--- DEBUG: Inspecting contents of gt_dir ---


NameError: name 'gt_dir' is not defined